# S02 — Ingesting Data Streams with Auto Loader

**Time: about 70 minutes.**
Covers: Ingesting Data Streams

### What you will be able to do afterwards

- Ingest files as they land with Auto Loader, exactly once.
- Control micro-batch size, and explain why that is your main throughput knob.
- Handle a column that appears mid-stream without losing data or the stream.
- Find the values that did not fit the schema instead of losing them silently.
- Deduplicate a stream that legitimately delivers the same record twice.

### The source

Twelve JSON files generated during setup. They are deterministic — the same events every
run — which is what lets the checks assert exact counts. They also contain three deliberate
defects:

| Defect | Where it comes from | Which step deals with it |
| :-- | :-- | :-- |
| Duplicate `event_id`s | each file replays a few events from the previous one | Step 5 |
| A new column, `referrer_url` | appears from file 009 onwards | Step 3 |
| Late-arriving timestamps | ~2% of events stamped ~25 min in the past | S03 |

Real feeds have all three. Yours has them on a schedule.

In [ ]:
from pyspark.sql import DataFrame, functions as F
from helpers import utils, event_stream, streaming_utils

cfg = utils.get_configs("web_events")
bronze_table, silver_table = cfg["table_bronze"], cfg["table_silver"]
checkpoint_path, schema_path = cfg["checkpoint_path"], cfg["schema_path"]
replay_path = event_stream.replay_path()

spark = utils.spark
manifest = event_stream.read_manifest()

print(f"source     {replay_path}")
print(f"bronze     {bronze_table}")
print(f"checkpoint {checkpoint_path}")
print(f"schema loc {schema_path}\n")
for k, v in manifest.items():
    print(f"  {k:24s} {v}")

## Step 1 — Inspect before you stream

**TO DO**

1. List the replay files.
2. Read **one early file** and **one late file** as a batch (`spark.read.json`) and compare
   their schemas.
3. Count the events in one file.

> **Question:** you can read these files perfectly well with `spark.read.json`. Give two
> concrete reasons to use Auto Loader instead, for this exact directory.

In [ ]:
# TO DO: list the files


# TO DO: read an early file and a late file as batch, compare schemas

## Step 2 — Stream them in, one file per micro-batch

**TO DO**

Write `read_events_stream()` and `write_events_stream()` and run them.

Read side:

- `cloudFiles.format = json`
- `cloudFiles.schemaLocation = schema_path`
- `cloudFiles.inferColumnTypes = true`
- `cloudFiles.maxFilesPerTrigger = 1`
- `cloudFiles.schemaEvolutionMode = addNewColumns`

Add an `_ingested_at` timestamp and `_source_file` from `_metadata.file_name`.

Write side: append to `bronze_table`, checkpoint at `checkpoint_path`,
`mergeSchema = true`, `trigger(availableNow=True)`, then `awaitTermination()`.

**Why `maxFilesPerTrigger = 1`.** Without it Auto Loader consumes all twelve files in one
micro-batch and you see a single commit. With it you get one micro-batch per file, so
`DESCRIBE HISTORY` becomes a readable log of the ingestion, watermarks advance step by step
in S03, and the whole run is reproducible. In production you would raise it — that is the
subject of S04.

**Expect this to stop early.** It will process files 001–008 and then fail when it meets
`referrer_url`. That is the designed behaviour of `addNewColumns`, and Step 3 is about it.

In [ ]:
def read_events_stream(source_path: str, schema_location: str) -> DataFrame:
    """
    Auto Loader stream over the replay directory, one file per trigger.

    Args:
        source_path: volume directory holding the JSON files.
        schema_location: where Auto Loader tracks the inferred schema.
    Returns:
        A streaming DataFrame with ingestion metadata added.
    """
    # TO DO
    pass


def write_events_stream(df: DataFrame, checkpoint: str, table: str):
    """Append the stream to a Delta table, returning the StreamingQuery."""
    # TO DO
    pass


# TO DO: run it, and expect it to stop when the schema changes

## Step 3 — Handle the schema change

**TO DO**

1. Read the exception. `query.exception()` or the cell output — it names the new column.
2. Look in `schema_path`. Auto Loader keeps a numbered schema per version; you should now
   see more than one. Print both and diff them.
3. Restart the **same** query, unchanged. It resumes from where it stopped and adopts the
   new column.
4. Repeat until all twelve files are consumed. Confirm the bronze row count matches
   `manifest["total_events"]`.
5. Check that `referrer_url` is null for events from the early files and populated for the
   later ones.

**Why stopping is the right default.** Adding a column changes the shape of every
downstream contract. `addNewColumns` stops so a human notices, records the new schema, and
resumes deliberately. `rescue` never stops but puts unexpected fields in a blob. `failOnNewColumns`
stops and does not record. `none` ignores the column entirely.

> **Questions:**
> - You restarted with no code change and it worked. What did the restart read to know what
>   to do?
> - Your pipeline runs at 3am unattended. Which evolution mode do you actually want, and
>   what has to exist around it for that to be a responsible choice?

In [ ]:
# TO DO: inspect the schema location — how many versions are there?


# TO DO: restart the stream until all files are consumed


# TO DO: verify the row count against the manifest, and check referrer_url

## Step 4 — Find what did not fit

`inferColumnTypes` guesses from the first files it sees. Anything that later fails to match
goes into the rescued-data column instead of being dropped.

**TO DO**

1. Find the rescued-data column in your bronze schema (default name `_rescued_data`).
2. Count how many rows have a non-null value.
3. `DESCRIBE HISTORY` on bronze — confirm one streaming commit per file.

> **Questions:**
> - If you had not set `cloudFiles.schemaLocation`, where would the rescued data have gone?
> - The rescued column is a string blob, not typed columns. Why is it stored that way, and
>   what would you do with a non-zero count in production?

In [ ]:
# TO DO: inspect the rescued data column


# TO DO: confirm one commit per file in the history

## Step 5 — Deduplicate into silver

Each file replays a few events from the previous one, so the same `event_id` genuinely
arrives twice.

**TO DO**

1. Create `silver_table` holding exactly one row per `event_id`.
2. `event_timestamp` must be a real `TIMESTAMP`, not a string — S03 depends on it.
3. Drop the ingestion-only columns you no longer need, but keep `_source_file`.
4. Confirm the row count equals `manifest["distinct_event_ids"]`.

You can do this as a batch read of bronze — a streaming dedup with watermarks comes in S03.
Choosing which copy to keep is a real decision: state your rule.

> **Question:** two copies of an event arrive with identical `event_id` but different
> `_ingested_at`. Which do you keep, and does the answer change if the payloads differ?

In [ ]:
# TO DO: build the deduplicated silver table


# TO DO: verify the count against the manifest

## Before you finish

In [ ]:
streaming_utils.stop_all_streams()

## Checks

In [ ]:
from helpers import test_runner

test_runner.run("S02-ingesting-streams-autoloader")

## Recap

- Auto Loader tracks consumed files in the checkpoint, which is where exactly-once
  ingestion comes from.
- `maxFilesPerTrigger` is the throughput knob and the reproducibility knob at once.
- `addNewColumns` stopping the stream is a feature. Restarting is the acknowledgement.
- The schema location is a versioned audit log of your source's shape. Read it during
  incidents.
- Rescued data is where values go when they do not fit. A non-zero count is a signal.